In [2]:
import os
import s3fs
import pandas as pd

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]},
    skip_instance_cache=True,
)

BUCKET = next(
    b for b in ["samuelbarre", "acharki-ensae"]
    if fs.exists(f"{b}/Projet-analyse-BPE/BPE25.parquet")
)

In [3]:
chemin_bpe = f"{BUCKET}/Projet-analyse-BPE/processed/bpe_alim_2025.parquet"
chemin_dvf = f"{BUCKET}/Projet-analyse-BPE/processed/dvf_2025.parquet"

with fs.open(chemin_bpe, "rb") as f:
    bpe = pd.read_parquet(f)

with fs.open(chemin_dvf, "rb") as f:
    dvf = pd.read_parquet(f)

print("BPE :", bpe.shape)
print("DVF :", dvf.shape)

BPE : (135587, 21)
DVF : (930120, 14)


In [4]:
chemin_insee = f"{BUCKET}/Projet-analyse-BPE/fichier_diffusion_2025.xlsx"

with fs.open(chemin_insee, "rb") as f:
    urbanisation = pd.read_excel(
        f,
        sheet_name="Maille communale",
        header=4,
        dtype={"CODGEO": str}
    )

urbanisation.head()

,CODGEO,LIBGEO,DENS,LIBDENS,PMUN22,P1,P2,P3,DENS_AAV,LIBDENS_AAV,DENS7,LIBDENS7
0,01001,L'Abergement-Clémenciat,3,Rural,859,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé
1,01002,L'Abergement-de-Varey,3,Rural,273,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé
2,01004,Ambérieu-en-Bugey,2,Urbain intermédiaire,15554,0.0,96.933762,3.066238,2,Urbain intermédiaire,2,Centres urbains intermédiaires
3,01005,Ambérieux-en-Dombes,3,Rural,1917,0.0,0.000000,100.000000,3,Rural périurbain,5,Bourgs ruraux
4,01006,Ambléon,3,Rural,114,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé


In [5]:
urbanisation = urbanisation[["CODGEO", "DENS", "LIBDENS"]]

urbanisation.head()

,CODGEO,DENS,LIBDENS
0,01001,3,Rural
1,01002,3,Rural
2,01004,2,Urbain intermédiaire
3,01005,3,Rural
4,01006,3,Rural


In [6]:
dvf["code_insee"] = dvf["code_commune"].copy()

# Regrouper les arrondissements dans leur commune
dvf.loc[dvf["code_insee"].between("75101", "75120"), "code_insee"] = "75056"
dvf.loc[dvf["code_insee"].between("69381", "69389"), "code_insee"] = "69123"
dvf.loc[dvf["code_insee"].between("13201", "13216"), "code_insee"] = "13055"

In [7]:
dvf = dvf.merge(
    urbanisation,
    left_on="code_insee",
    right_on="CODGEO",
    how="left",
    validate="many_to_one"
).drop(columns="CODGEO")

In [8]:
dvf.head(10)

,id_mutation,date_mutation,nature_mutation,valeur_fonciere,code_commune,nom_commune,code_departement,type_local,surface_reelle_bati,surface_terrain,longitude,latitude,prix_m2,geometry,code_insee,DENS,LIBDENS
0,2025-1,2025-01-07,Vente,468000.0,01158,Farges,01,Maison,111,133.0,5.907199,46.170775,4216.216216,"b'\x01\x01\x00\x00\x00Hi\xeb\xaa\xaf4,Amz\xa7\...",01158,3,Rural
1,2025-2,2025-01-06,Vente,180000.0,01257,Montanges,01,Appartement,87,76.0,5.801490,46.164545,2068.965517,b'\x01\x01\x00\x00\x00\x1a\xd3\xb5\xd33\xf5+A\...,01257,3,Rural
2,2025-3,2025-01-06,Vente,490000.0,01065,Buellas,01,Maison,190,500.0,5.155043,46.206188,2578.947368,b'\x01\x01\x00\x00\x00\xc0\xe4\xc1%\xb7n*A[*i\...,01065,3,Rural
3,2025-4,2025-01-09,Vente,183000.0,01173,Gex,01,Appartement,35,NaN,6.064268,46.332953,5228.571429,"b'\x01\x01\x00\x00\x00\x91\x84\x0fo\xd5\x8d,A\...",01173,2,Urbain intermédiaire
4,2025-5,2025-01-07,Vente,295000.0,01028,Baneins,01,Maison,95,965.0,4.910721,46.121708,3105.263158,b'\x01\x01\x00\x00\x00\xf4\xac\x08\xb7]\xdd)A\...,01028,3,Rural
5,2025-7,2025-01-08,Vente,131937.3,01005,Amberieux-en-Dombes,01,Appartement,49,NaN,4.905917,45.995081,2692.597959,b'\x01\x01\x00\x00\x00\x8dw\xf7\xc9\x1e\xdd)A]...,01005,3,Rural
6,2025-8,2025-01-09,Vente,173000.0,01306,Pont-de-Veyle,01,Appartement,78,NaN,4.887005,46.260695,2217.948718,b'\x01\x01\x00\x00\x00E\x11Mh.\xcc)A\xd9<\r>\x...,01306,3,Rural
7,2025-9,2025-01-06,Vente,425000.0,01318,Rance,01,Maison,110,500.0,4.870114,45.973288,3863.636364,b'\x01\x01\x00\x00\x00\xfa\x17r\xab\xec\xc7)A\...,01318,3,Rural
8,2025-10,2025-01-09,Vente,1165000.0,01419,Thoiry,01,Maison,190,860.0,5.981172,46.243363,6131.578947,"b'\x01\x01\x00\x00\x00\xf6\xc1|\x96\xdd^,A\xdf...",01419,2,Urbain intermédiaire
9,2025-11,2025-01-07,Vente,630000.0,01308,Pougny,01,Maison,100,761.0,5.950677,46.136450,6300.000000,"b'\x01\x01\x00\x00\x00y\\\xcc\xa5\xfcO,As\xa7\...",01308,3,Rural


In [9]:
print(dvf["LIBDENS"].value_counts(dropna=False))

LIBDENS
Urbain dense            340141
Rural                   311352
Urbain intermédiaire    278627
Name: count, dtype: int64


In [10]:
import numpy as np
from sklearn.neighbors import BallTree

# Rayon de la Terre en m
R = 6371000

coord_dvf = np.radians(dvf[["latitude", "longitude"]].to_numpy())
coord_bpe = np.radians(bpe[["LATITUDE", "LONGITUDE"]].to_numpy())

# Construire l'arbre des commerces
arbre = BallTree(coord_bpe, metric="haversine")

dvf["commerces_500m"] = arbre.query_radius(
    coord_dvf,
    r=500 / R,
    count_only=True
)

distances, indices = arbre.query(coord_dvf, k=1)
dvf["distance_commerce_m"] = distances[:, 0] * R

dvf[["commerces_500m", "distance_commerce_m"]].head(10)

,commerces_500m,distance_commerce_m
0,0,870.463654
1,0,2303.620286
2,0,1093.225261
3,3,274.800147
4,0,1935.882139
5,3,74.260125
6,4,42.104244
7,0,3547.864870
8,1,296.631891
9,0,2951.043402


In [11]:
dvf[["commerces_500m", "distance_commerce_m"]].describe()

,commerces_500m,distance_commerce_m
count,930120.000000,9.301200e+05
mean,12.714714,6.255749e+02
std,26.384550,3.884879e+03
min,0.000000,3.640851e-02
25%,0.000000,9.969786e+01
50%,3.000000,2.371424e+02
75%,11.000000,5.678347e+02
max,283.000000,3.606995e+06


In [12]:
dvf.nlargest(10, "distance_commerce_m")[
    ["nom_commune", "code_departement",
     "distance_commerce_m", "commerces_500m"]
]

,nom_commune,code_departement,distance_commerce_m,commerces_500m
856516,Issy-les-Moulineaux,92,3.606995e+06,0
856872,Vaucresson,92,1.691870e+05,0
855956,Vaucresson,92,1.690181e+05,0
371022,Bourriot-Bergonce,40,1.441620e+04,0
370042,Bourriot-Bergonce,40,1.436034e+04,0
374375,Bourriot-Bergonce,40,1.435374e+04,0
369590,Bourriot-Bergonce,40,1.426223e+04,0
459039,Saint-Ouen-Domprot,51,1.341388e+04,0
459272,Saint-Ouen-Domprot,51,1.324008e+04,0
583023,Borce,64,1.289183e+04,0


In [13]:
dvf.nlargest(10, "distance_commerce_m")[
    ["nom_commune", "longitude", "latitude",
     "distance_commerce_m"]
]

,nom_commune,longitude,latitude,distance_commerce_m
856516,Issy-les-Moulineaux,0.269463,83.517658,3.606995e+06
856872,Vaucresson,2.287487,40.882084,1.691870e+05
855956,Vaucresson,2.285057,40.883919,1.690181e+05
371022,Bourriot-Bergonce,-0.186952,44.121401,1.441620e+04
370042,Bourriot-Bergonce,-0.160891,44.116882,1.436034e+04
374375,Bourriot-Bergonce,-0.191450,44.124067,1.435374e+04
369590,Bourriot-Bergonce,-0.198201,44.152456,1.426223e+04
459039,Saint-Ouen-Domprot,4.412537,48.609575,1.341388e+04
459272,Saint-Ouen-Domprot,4.407830,48.610175,1.324008e+04
583023,Borce,-0.557828,42.857609,1.289183e+04


Erreur :
Issy-les-Moulineaux latitude = 48°49′26″N
Vaucresson latitude = 48.83

On retire les 3 valeurs absurdes (856516, 856872 & 855956)

In [14]:
dvf_reg = dvf.drop(index=[856516, 856872, 855956]).copy()


In [16]:
import statsmodels.formula.api as smf

dvf_reg["log_prix_m2"] = np.log(dvf_reg["prix_m2"])

modele1 = smf.ols(
    "log_prix_m2 ~ commerces_500m + surface_reelle_bati + C(type_local) + C(code_departement)",
    data=dvf_reg
).fit(cov_type="HC3")

print(modele1.summary())

                            OLS Regression Results                            
Dep. Variable:            log_prix_m2   R-squared:                       0.402
Model:                            OLS   Adj. R-squared:                  0.402
Method:                 Least Squares   F-statistic:                     7426.
Date:                Wed, 11 Nov 2026   Prob (F-statistic):               0.00
Time:                        15:06:52   Log-Likelihood:            -8.3127e+05
No. Observations:              930117   AIC:                         1.663e+06
Df Residuals:                  930021   BIC:                         1.664e+06
Df Model:                          95                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [17]:
tableau = modele1.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.152305,0.006495,1255.247368,0.0,8.139576,8.165034
C(type_local)[T.Maison],-0.095077,0.001862,-51.074315,0.0,-0.098726,-0.091429
commerces_500m,0.001837,0.000035,52.016814,0.0,0.001767,0.001906
surface_reelle_bati,-0.002245,0.000027,-82.776999,0.0,-0.002298,-0.002192


Toutes choses égales par ailleurs, une maison présente en moyenne un prix au m² environ 9,5 % inférieur à celui d'un appartement.
Toutes choses égales par ailleurs, un commerce alimentaire supplémentaire dans un rayon de 500 mètres est associé en moyenne à une augmentation d'environ 0,18 % du prix au m².
Ces deux effets sont statistiquements significatifs au seuil de 1 %.

In [18]:
modele2 = smf.ols(
    "log_prix_m2 ~ commerces_500m * C(DENS) + surface_reelle_bati + C(type_local) + C(code_departement)",
    data=dvf_reg
).fit(cov_type="HC3")

In [19]:
tableau = modele2.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", modele2.rsquared)
print("Observations :", modele2.nobs)

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.232003,0.006448,1276.668172,0.000000e+00,8.219365,8.244641
C(DENS)[T.2],-0.022467,0.001941,-11.577071,5.385480e-31,-0.026270,-0.018663
C(DENS)[T.3],-0.243982,0.002199,-110.932942,0.000000e+00,-0.248293,-0.239671
C(type_local)[T.Maison],-0.026947,0.001961,-13.744100,5.525576e-43,-0.030790,-0.023104
commerces_500m,0.001266,0.000036,35.077176,1.502433e-269,0.001196,0.001337
commerces_500m:C(DENS)[T.2],-0.001203,0.000131,-9.175560,4.492329e-20,-0.001459,-0.000946
commerces_500m:C(DENS)[T.3],0.012312,0.000451,27.299591,4.289271e-164,0.011428,0.013196
surface_reelle_bati,-0.002217,0.000027,-82.074690,0.000000e+00,-0.002269,-0.002164


R² : 0.41376965772746876
Observations : 930117.0


In [20]:
print(modele2.t_test("commerces_500m + commerces_500m:C(DENS)[T.2] = 0"))
print(modele2.t_test("commerces_500m + commerces_500m:C(DENS)[T.3] = 0"))

                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0          6.387e-05      0.000      0.500      0.617      -0.000       0.000
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.0136      0.000     30.134      0.000       0.013       0.014


En zone urbaine dense, un commerce alimentaire supplémentaire dans un rayon de 500 mètres est associé à une augmentation du prix au m² d’environ 0,13 %, ce qui reste relativement faible.

En zone urbaine intermédiaire, cet effet est pratiquement nul (0,001266 − 0,001203 = 0,0063 %) et n’est pas statistiquement significatif aux seuils usuels (p-value = 0,617).

En revanche, en zone rurale, un commerce supplémentaire est associé à une augmentation du prix au m² d’environ 1,36 % (0,001266 + 0,012312 = 0,013578), ce qui représente un effet économiquement plus important.

Les associations estimées sont statistiquement significatives au seuil de 1 % en zones urbaine dense et rurale, mais pas en zone urbaine intermédiaire. Ces résultats suggèrent donc que la proximité des commerces alimentaires est davantage associée aux prix immobiliers dans les zones rurales.

In [21]:
modele3 = smf.ols(
    "log_prix_m2 ~ distance_commerce_m * C(DENS) + surface_reelle_bati + C(type_local) + C(code_departement)",
    data=dvf_reg
).fit(cov_type="HC3")

In [22]:
tableau = modele3.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", modele3.rsquared)
print("Observations :", modele3.nobs)

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.236187,0.006516,1264.007615,0.000000e+00,8.223416,8.248958
C(DENS)[T.2],-0.042923,0.002446,-17.548173,6.142413e-69,-0.047717,-0.038129
C(DENS)[T.3],-0.160742,0.002691,-59.733379,0.000000e+00,-0.166016,-0.155468
C(type_local)[T.Maison],-0.042485,0.001940,-21.896667,2.794904e-106,-0.046288,-0.038682
distance_commerce_m,0.000093,0.000006,15.937364,3.487589e-57,0.000082,0.000105
distance_commerce_m:C(DENS)[T.2],-0.000039,0.000006,-6.172986,6.701229e-10,-0.000052,-0.000027
distance_commerce_m:C(DENS)[T.3],-0.000137,0.000006,-23.129693,2.327811e-118,-0.000148,-0.000125
surface_reelle_bati,-0.002256,0.000027,-83.259753,0.000000e+00,-0.002309,-0.002203


R² : 0.41477338003489095
Observations : 930117.0


In [23]:
print(modele3.t_test("distance_commerce_m + distance_commerce_m:C(DENS)[T.2] = 0"))

print(modele3.t_test("distance_commerce_m + distance_commerce_m:C(DENS)[T.3] = 0"))

                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0          5.424e-05   2.75e-06     19.752      0.000    4.89e-05    5.96e-05
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0         -4.319e-05   8.46e-07    -51.054      0.000   -4.48e-05   -4.15e-05


En zone urbaine dense, une augmentation de 100 mètres de la distance au commerce alimentaire le plus proche est associée à une hausse du prix au m² d'environ 0,93 %, toutes choses égales par ailleurs. Cet effet est statistiquement significatif au seuil de 1 %.

En zone urbaine intermédiaire, cette association reste positive, mais plus faible : 0,000093 − 0,000039 = 0,000054, soit une hausse d'environ 0,54 % pour 100 mètres supplémentaires.

En revanche, en zone rurale, la relation s'inverse : 0,000093 − 0,000137 = −0,000044, soit une baisse du prix au m² d'environ 0,44 % pour 100 mètres supplémentaires.

Les coefficients d'interaction sont statistiquement significatifs au seuil de 1 %, ainsi la relation entre la distance aux commerces et les prix immobiliers est significativement différente selon le degré d'urbanisation.

In [24]:
dvf_reg["log_distance"] = np.log(dvf_reg["distance_commerce_m"])

modele3_log = smf.ols(
    "log_prix_m2 ~ log_distance * C(DENS) + surface_reelle_bati + C(type_local) + C(code_departement)",
    data=dvf_reg
).fit(cov_type="HC3")

In [25]:

tableau = modele3_log.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", modele3_log.rsquared)
print("Observations :", modele3_log.nobs)

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.230857,0.008107,1015.248698,0.000000e+00,8.214967,8.246747
C(DENS)[T.2],-0.146668,0.007913,-18.536045,1.057153e-76,-0.162176,-0.131160
C(DENS)[T.3],-0.092329,0.007928,-11.646289,2.396699e-31,-0.107867,-0.076791
C(type_local)[T.Maison],-0.046327,0.001956,-23.678666,5.982288e-124,-0.050161,-0.042492
log_distance,0.005796,0.001006,5.760772,8.372993e-09,0.003824,0.007769
log_distance:C(DENS)[T.2],0.019239,0.001445,13.312278,1.963996e-40,0.016406,0.022071
log_distance:C(DENS)[T.3],-0.023156,0.001334,-17.358081,1.713554e-67,-0.025771,-0.020542
surface_reelle_bati,-0.002263,0.000027,-83.346518,0.000000e+00,-0.002316,-0.002210


R² : 0.41286011422529345
Observations : 930117.0


In [26]:
print(modele3_log.t_test("log_distance + log_distance:C(DENS)[T.2] = 0"))
print(modele3_log.t_test("log_distance + log_distance:C(DENS)[T.3] = 0"))

                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.0250      0.001     22.847      0.000       0.023       0.027
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.0174      0.001    -19.136      0.000      -0.019      -0.016


Contrairement au modèle précédent, qui mesurait l'effet d'une augmentation de la distance en mètres sur le prix au m², ce modèle (log-log) mesure l'élasticité du prix au m² par rapport à la distance au commerce le plus proche. Autrement dit, on regarde la variation en pourcentage du prix au m² associée à une augmentation de 1 % de la distance. Ainsi, doubler la distance de 100 à 200 mètres revient ici à la doubler de 1 000 à 2 000 mètres.

En zone urbaine dense, une augmentation de 1 % de la distance au commerce le plus proche est associée à une hausse du prix au m² d'environ 0,0058 %, toutes choses égales par ailleurs. Cette association est statistiquement significative au seuil de 1 %, mais reste relativement faible.

En zone urbaine intermédiaire, cette association est plus importante (0,005796 + 0,019239 = 0,025035), soit une hausse du prix au m² d'environ 0,025 % pour 1 % de distance supplémentaire.

En revanche, en zone rurale, la relation est négative (0,005796 − 0,023156 = −0,017360) : une augmentation de 1 % de la distance est associée à une baisse du prix au m² d'environ 0,017 %.


Interprétations possibles (hypothèses mais pas une relation causale) :

Dans les zones urbaines, l'éloignement des commerces est associé à des prix immobiliers plus élevés. Une explication possible serait que les logements plus éloignés des commerces bénéficient d'un environnement plus résidentiel, moins bruyant ou moins fréquenté.

Dans les zones rurales, l'éloignement des commerces est associé à des prix plus faibles. Cela pourrait montrer l'importance de l'accessibilité aux services dans des territoires où ils sont moins nombreux.

Les deux derniers modèles ont des R² quasiment identiques. Il est difficile de choisir lequel privilégier : le modèle log-niveau qui mesure l'effet d'une augmentation de la distance en mètres, ou le modèle log-log qui mesure l'effet d'une augmentation de la distance en pourcentage (élasticité).